# SupplyPulse AI — Multi-Target Model Training & Optuna Tuning

This notebook demonstrates the construction of production Scikit-Learn Pipelines with `ColumnTransformer` and Bayesian Hyperparameter Optimization with Optuna.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
import joblib
import os
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score, f1_score

## 1. Pipeline Formulation

In [ ]:
def build_pipeline(numeric_cols, categorical_cols, model):
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_cols),
            ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), categorical_cols)
        ]
    )
    return Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])

print("Pipeline factory configured.")

## 2. Bayesian Hyperparameter Search Space (Optuna)

In [ ]:
def sample_xgb_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 80, 100),
        "max_depth": trial.suggest_int("max_depth", 4, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.05, 0.1),
        "subsample": 0.8,
        "random_state": 42
    }
    return params

study = optuna.create_study(direction="maximize")
print("Optuna Bayesian optimization study initialized.")

## 3. Verified Production Hyperparameters

Extracted from `experiments_history.json` across all 8 operational models:

In [ ]:
exp_path = os.path.join("..", "models", "experiments_history.json")
if not os.path.exists(exp_path):
    exp_path = os.path.join("models", "experiments_history.json")

import json
with open(exp_path, "r") as f:
    exps = json.load(f)

hparams_summary = []
for e in exps:
    if "target" in e:
        hparams_summary.append({
            "Target": e["target"],
            "Architecture": e["model_architecture"],
            "Estimators": e["hyperparameters"].get("n_estimators", "N/A"),
            "Depth": e["hyperparameters"].get("max_depth", "N/A"),
            "Learning Rate": round(e["hyperparameters"].get("learning_rate", 0), 4) if "learning_rate" in e["hyperparameters"] else "N/A"
        })
pd.DataFrame(hparams_summary)